In [1]:
from tokenizers import Tokenizer, decoders
from transformers import AutoTokenizer
from datasets import load_dataset

In [2]:
import json, pickle, os
from tqdm.notebook import tqdm
import unicodedata
from math import ceil
from collections import Counter

In [3]:
import tokenizer_edge_cases as ec

In [4]:
model_dir = "/Users/nirajh/Code/Language_adaptation/olmo2-7b-stage1"

#### Loading the tokenizer and modifying its regex pattern

In [5]:
# Loading the tokenizer
tok = AutoTokenizer.from_pretrained(model_dir)

In [6]:
## Changing the regex pattern in the pre tokenizer, so that kannada words are split correctly
tj = json.loads(tok.backend_tokenizer.to_str())        # Getting the tokenizer's vocab, merges, special tokens, and pre-processing configs
split = tj["pre_tokenizer"]["pretokenizers"][0]        # Fetches the first pre-tokenizer (they are usually chained together under a Sequence)
old = split["pattern"]["Regex"]
new = old.replace(r"[^\r\n\p{L}\p{N}]?\p{L}+", r"[^\r\n\p{L}\p{N}]?[\p{L}\p{M}]+")
assert new != old, "pattern not found; print(old) and adjust"
split["pattern"]["Regex"] = new

In [7]:
# Checking its output
kn_tok = Tokenizer.from_str(json.dumps(tj))
text = "ನಮಸ್ಕಾರ ಕನ್ನಡ ಭಾಷೆಯಲ್ಲಿ"
print([text[s:e] for _, (s, e) in kn_tok.pre_tokenizer.pre_tokenize_str(text)])

['ನಮಸ್ಕಾರ', ' ಕನ್ನಡ', ' ಭಾಷೆಯಲ್ಲಿ']


#### Loading the new merges list

In [8]:
# Loading the new merges list
final_path = "/Users/nirajh/Code/Language_adaptation/merges_files/new_merges.pkl"

with open(final_path, "rb") as f:
    merges_ = pickle.load(f)

#### Loading the eval set

In [9]:
kn_texts = []

with open("/Users/nirajh/Code/Language_adaptation/eval/kn_eval_texts.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        kn_texts.append(data["text"])

print(f"Successfully read {len(kn_texts)} lines.")   # Should be 1997

Successfully read 1997 lines.


In [10]:
en_texts = []

with open("/Users/nirajh/Code/Language_adaptation/eval/en_eval_texts.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        en_texts.append(data["text"])

print(f"Successfully read {len(en_texts)} lines.")   # Should be 2000 lines

Successfully read 2000 lines.


In [11]:
bpe_model = kn_tok.model
bytelevel = decoders.ByteLevel()

In [12]:
def is_kannada(w):
    return "à²" in w or "à³" in w


def hybrid_pieces(text, u):
    text = unicodedata.normalize("NFC", text)
    pieces = []
    for w, _ in kn_tok.pre_tokenizer.pre_tokenize_str(text):        # This pre-tokenizes to byte-level chunks, which is why the above is_kannada function is looking for byte chars
        model = u.model if is_kannada(w) else bpe_model             # Sends Kannada chunks to the Unigram model and everything else to OLMo's BPE
        pieces.extend(t.value for t in model.tokenize(w))
    return pieces

In [13]:
def hybrid_stats(u, texts, lang):
    n_tok = n_words = n_chars = n_frag = 0
    for t in tqdm(texts, leave=False, desc=f"Tokenizer stats on {lang} eval set"):
        pieces = hybrid_pieces(t, u)
        n_tok += len(pieces)
        n_words += len(t.split())
        n_chars += len(t)
        n_frag += sum("\ufffd" in bytelevel.decode([p]) for p in pieces)
    return {
        "fertility (tokens/word)": n_tok / n_words,
        "chars/token": n_chars / n_tok,
        "fragment rate": n_frag / n_tok,
    }

#### Calculating stats for unigram model with vocab size 4000

In [14]:
out_dir = "/Users/nirajh/Code/Language_adaptation/tokenizers/unigram"
tok_4000 = Tokenizer.from_file(os.path.join(out_dir, "unigram_4000.json"))

In [15]:
# Counting the unigram tokens to find the most frequent ones (in the eval set)
uni_counts = Counter()

for t in kn_texts:
    t = unicodedata.normalize("NFC", t)
    for w, _ in kn_tok.pre_tokenizer.pre_tokenize_str(t):
        if is_kannada(w):
            uni_counts.update(p.value for p in tok_4000.model.tokenize(w))

In [16]:
# Viewing the 25 most common unigram tokens
total = sum(uni_counts.values())

for p, c in uni_counts.most_common(25):
    print(f"{bytelevel.decode([p])!r:20} {c:>8,}  ({c / total:.2%})")

' �'                   57,759  (5.30%)
' '                    23,645  (2.17%)
'ದ'                    14,661  (1.34%)
'ಗಳ'                   10,393  (0.95%)
'ರ'                     9,269  (0.85%)
'ರು'                    8,504  (0.78%)
'ಯ'                     8,098  (0.74%)
'ನ್ನು'                  7,429  (0.68%)
'ದಲ್ಲಿ'                 7,054  (0.65%)
'ನ'                     6,652  (0.61%)
'ಿದ್ದ'                  6,527  (0.60%)
'ು'                     6,088  (0.56%)
'�'                     6,020  (0.55%)
'ುತ್ತ'                  5,839  (0.54%)
' ಈ'                    5,442  (0.50%)
'ಗೆ'                    5,384  (0.49%)
'ಗಳು'                   5,318  (0.49%)
'ವ'                     5,255  (0.48%)
' ಮತ್ತು'                5,085  (0.47%)
'ಿ'                     4,778  (0.44%)
'ಿದೆ'                   4,757  (0.44%)
'ಲ'                     4,735  (0.43%)
' ಅ'                    4,531  (0.42%)
'ರ್'                    4,460  (0.41%)
'ಕ್ಕೆ'                  4,405  (0.40%)


The most frequent tokens are Kannada characters and suffixes. The first 25 merges that the BPE tokenizer learnt were mostly Kannada characters and a few ottaksharas.

In [17]:
# Below code shows what the model sees as its top 25 pieces. This doesn't have the is_kannada() filter. The results are very similar

# uni_vocab = json.loads(tok_4000.to_str())["model"]["vocab"]     # list of [piece, score]
# for p, score in sorted(uni_vocab, key=lambda x: -x[1])[:25]:
#     print(f"{bytelevel.decode([p])!r:20} {score:8.3f}")

In [18]:
kn_stats_4k = hybrid_stats(tok_4000, kn_texts, "Kannada")
en_stats_4k = hybrid_stats(tok_4000, en_texts, "English")

Tokenizer stats on Kannada eval set:   0%|          | 0/1997 [00:00<?, ?it/s]

Tokenizer stats on English eval set:   0%|          | 0/2000 [00:00<?, ?it/s]

In [19]:
print("Stats on Kannada eval set: \n",*kn_stats_4k.items(), sep='\n')
print("\n\nStats on English eval set: \n",*en_stats_4k.items(), sep='\n')

Stats on Kannada eval set: 

('fertility (tokens/word)', 2.60124139182775)
('chars/token', 3.1077979567198475)
('fragment rate', 0.1374849090649462)


Stats on English eval set: 

('fertility (tokens/word)', 1.2804896737381426)
('chars/token', 4.7985698471112705)
('fragment rate', 0.0013623353179945945)


#### Calculating stats for unigram model with vocab size 6000

In [20]:
tok_6000 = Tokenizer.from_file(os.path.join(out_dir, "unigram_6000.json"))

In [21]:
# Getting the original vocabulary of the OLMo tokenizer
orig_vocab = tok.get_vocab()

# Fetching the tokens that are in the Unigram tokenizer's vocab, but are not present in OLMo's original tokenizer
u_new_tokens_6k = ec.unigram_new_tokens(tok_6000, orig_vocab)
print("Number of new Kannada tokens in the Unigram tokenizer:", len(u_new_tokens_6k))

Number of new Kannada tokens in the Unigram tokenizer: 5357


In [22]:
kn_stats_6k = hybrid_stats(tok_6000, kn_texts, "Kannada")
en_stats_6k = hybrid_stats(tok_6000, en_texts, "English")

Tokenizer stats on Kannada eval set:   0%|          | 0/1997 [00:00<?, ?it/s]

Tokenizer stats on English eval set:   0%|          | 0/2000 [00:00<?, ?it/s]

In [23]:
print("Stats on Kannada eval set: \n",*kn_stats_6k.items(), sep='\n')
print("\n\nStats on English eval set: \n",*en_stats_6k.items(), sep='\n')

Stats on Kannada eval set: 

('fertility (tokens/word)', 2.451715091922511)
('chars/token', 3.2973377327127373)
('fragment rate', 0.16338592674912777)


Stats on English eval set: 

('fertility (tokens/word)', 1.2804896737381426)
('chars/token', 4.7985698471112705)
('fragment rate', 0.0013623353179945945)
